# 8.3: Network in Network (NiN)

In [6]:
import torch
from torch import nn
from d2l import torch as d2l

## 8.3.1: NiN Blocks

In [7]:
def nin_block(out_channels, kernel_size, strides, padding):
    return nn.Sequential(
        nn.LazyConv2d(out_channels, kernel_size, strides, padding), nn.ReLU(),
        nn.LazyConv2d(out_channels, kernel_size=1), nn.ReLU(), # 1x1 kernel
        nn.LazyConv2d(out_channels, kernel_size=1), nn.ReLU()  # 1x1 kernel
    )

## 8.3.2: NiN Model

In [8]:
class NiN(d2l.Classifier):
    def __init__(self, lr=0.1, num_classes=10):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(
            nin_block(96, kernel_size=11, strides=4, padding=0),
            nn.MaxPool2d(3, stride=2),
            nin_block(256, kernel_size=5, strides=1, padding=2),
            nn.MaxPool2d(3, stride=2),
            nin_block(384, kernel_size=3, strides=1, padding=1),
            nn.MaxPool2d(3, stride=2),
            nn.Dropout(0.5),
            nin_block(num_classes, kernel_size=3, strides=1, padding=1),
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten())
        self.net.apply(d2l.init_cnn)

In [9]:
NiN().layer_summary((1,1,224,224)) # View output shape of every block using an example

Sequential output shape:	 torch.Size([1, 96, 54, 54])
MaxPool2d output shape:	 torch.Size([1, 96, 26, 26])
Sequential output shape:	 torch.Size([1, 256, 26, 26])
MaxPool2d output shape:	 torch.Size([1, 256, 12, 12])
Sequential output shape:	 torch.Size([1, 384, 12, 12])
MaxPool2d output shape:	 torch.Size([1, 384, 5, 5])
Dropout output shape:	 torch.Size([1, 384, 5, 5])
Sequential output shape:	 torch.Size([1, 10, 5, 5])
AdaptiveAvgPool2d output shape:	 torch.Size([1, 10, 1, 1])
Flatten output shape:	 torch.Size([1, 10])


In [ ]:
model = NiN(lr=0.5)
trainer = d2l.Trainer(max_epochs=10, num_gpus=1)
data = d2l.FashionMNIST(batch_size=128, resize=(224,224))
model.apply_init([next(iter(data.get_dataloader(True)))[0]], d2l.init_cnn)
trainer.fit(model,data)

# Summary

* NiN has dramatically fewer parameters than AlexNet and VGG, primarily due to fact that it needs no giant fully connected layers
* NiN uses global average pooling to aggregate across all images locations after last stage of network body, replacing expenseive learned reduciton operations with a simple average
* Averaging across a low-resolution representation (with many channels) also adds to the amount of translation invariance the network can handle
* NiN chooses fewer convolutions with wider kernels, replacing them with $1 \times 1$ convolutions. These convolutions can cater for a significant amount of nonlinearity across channels within any given locations
* Both $1 \times 1$ convolutions and global average pooling significantly influenced subsequent CNN designs.